# 01 · 环境快速上手与数据初探

用途：**一页纸确认"环境搭好了 + 数据能取"**，然后进 `02_forecast_baseline` 做正题。

- 环境自检：Python 版本（赛题硬约束 3.10/3.11）、依赖版本、关键 API import；
- 数据初探：`data_loader` 试用——**长表**（date/store_id/item_id/demand）合成数据
  （本次实验数据来源，参数对齐 config 基准 4 SKU × 365 天）/ M5 子集（加载路径已实现，本次未启用）；
- 交接对象：`ReplayDemandGenerator`（需求交接对象）与 `pivot_demand_wide` 长表→宽表；
- 快速可视化：图落盘 `outputs/figures/`（绘图函数来自 `src/visualization/dashboard.py`）。

> 本 notebook 只涉及数据与预测模块：不读取 `config/network_config.yaml` 的内容，
> 不涉及拓扑/补货/协同逻辑。

In [ ]:
# ---- 环境自检（三项：Python 版本 / 依赖版本 / 关键 API 导入）----
import sys
from pathlib import Path

_root = Path.cwd()
if _root.name == "notebooks":          # 兼容内核工作目录在 notebooks/ 的情况
    _root = _root.parent
sys.path.insert(0, str(_root))

import importlib

# 1) Python 版本：赛题硬约束 3.10 / 3.11（本机默认 3.13 不合规，须用 .venv）
v = sys.version_info
py_ok = (v.major, v.minor) in {(3, 10), (3, 11)}
print(f"[{'PASS' if py_ok else 'FAIL'}] Python {v.major}.{v.minor}.{v.micro}"
      f"（要求 3.10/3.11）")
print(f"        解释器: {sys.executable}"
      + ("" if py_ok else "\n        [警告] 版本不合规，请用项目 .venv 内核重跑！"))

# 2) 依赖版本
for pkg in ["numpy", "pandas", "matplotlib", "scipy", "deepbullwhip"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"[PASS] {pkg:<14s} {getattr(mod, '__version__', '?')}")
    except Exception as exc:                       # noqa: BLE001
        print(f"[FAIL] {pkg:<14s} 导入失败: {exc!r}")

# 3) 关键 API 导入（A/B 后续开发的地基；注意 OrderingPolicy 顶层未导出）
CRITICAL_IMPORTS = [
    ("deepbullwhip", "EchelonConfig"),
    ("deepbullwhip", "EdgeConfig"),
    ("deepbullwhip", "SupplyChainGraph"),
    ("deepbullwhip", "NetworkSupplyChain"),
    ("deepbullwhip", "SerialSupplyChain"),
    ("deepbullwhip", "Forecaster"),
    ("deepbullwhip", "ReplayDemandGenerator"),
    ("deepbullwhip.policy.base", "OrderingPolicy"),           # 顶层未导出
    ("deepbullwhip.policy.order_up_to", "OrderUpToPolicy"),
    ("deepbullwhip.datasets", "load_dataset"),
    ("deepbullwhip.datasets", "load_m5"),
]
n_ok = 0
for module, attr in CRITICAL_IMPORTS:
    try:
        getattr(importlib.import_module(module), attr)
        n_ok += 1
        print(f"[PASS] {module}.{attr}")
    except Exception as exc:                       # noqa: BLE001
        print(f"[FAIL] {module}.{attr} -> {exc!r}")

print(f"\n环境自检：关键 API {n_ok}/{len(CRITICAL_IMPORTS)} 通过。"
      "全部 PASS 即可进 02_forecast_baseline。")

In [ ]:
# ---- data_loader 试用：合成数据（演示默认）/ M5 子集（代码保留）----
# 【合成数据】框架 AR(1)，默认参数对齐 config/network_config.yaml 基准
#   （sku_count=5、simulation_days=365、demand.mean=10、demand.std=3），seed 可复现。
# 【M5 真实数据】需要 data/raw/m5/sales_train_evaluation.csv（本地放置，不联网），
#   daily 口径（截取前 365 天）；两套口径同为长表 date/store_id/item_id/demand。
import numpy as np
import pandas as pd

from src.data_loader import (
    load_m5_demand, load_synthetic_demand, build_replay_generator, pivot_demand_wide)

USE_M5 = False   # <-- 拿到 M5 本地 CSV 后改为 True
STORE, DEPTS = "CA_1", ["FOODS_1", "FOODS_2", "HOUSEHOLD_1", "HOUSEHOLD_2"]

if USE_M5:
    try:
        demand_df = load_m5_demand(STORE, DEPTS)
        data_source = f"M5 {STORE} x{len(DEPTS)}"
        print(f"[M5] 加载成功：{len(DEPTS)} 条 dept 级序列（daily，前 365 天）")
    except (FileNotFoundError, ImportError, ValueError) as exc:
        # M5 不可用时不静默：打印原因后降级合成
        print(f"[M5] 不可用（{type(exc).__name__}），降级合成：{str(exc).splitlines()[0]}")
        demand_df = load_synthetic_demand()
        data_source = "synthetic AR(1) (M5 fallback)"
else:
    demand_df = load_synthetic_demand()
    data_source = "synthetic AR(1)"
    print("[合成] 使用框架 AR(1) 合成需求（对齐 config：5 SKU x 365 天、mean=10、std=3）")

print(f"数据来源: {data_source} | 长表形状 (行, 列): {demand_df.shape} "
      f"| NaN 总数: {int(demand_df.isna().sum().sum())}")
print(f"期数（unique date）: {demand_df['date'].nunique()} | "
      f"序列数（unique item_id）: {demand_df['item_id'].nunique()}")
demand_df.describe().round(2)     # 各序列均值/分位数一览（M5 口径下是真实销量单位）

In [ ]:
# ---- 需求交接对象：build_replay_generator ----
# 整表（长表）按 date 聚合求和成单序列；逐节点按 item_id 过滤后各自构造——
# B 拓扑里给每个终端门店配 generator 就用后一种方式。
from deepbullwhip.demand.replay import ReplayDemandGenerator

replay_gen = build_replay_generator(demand_df)
assert isinstance(replay_gen, ReplayDemandGenerator)
print(f"整表聚合回放长度 {len(replay_gen.data)}；"
      f"generate(12) 前 6 期: {np.round(replay_gen.generate(T=12)[:6], 2)}")

node_generators = {f"{s}_{i}": build_replay_generator(g)
                   for (s, i), g in demand_df.groupby(["store_id", "item_id"],
                                                      sort=True)}
assert all(isinstance(g, ReplayDemandGenerator) for g in node_generators.values())
print(f"逐节点 generator: { {k: len(g.data) for k, g in node_generators.items()} }")

In [ ]:
# ---- 快速可视化：需求总览图（绘图函数来自 src/visualization/dashboard.py，长表自动 pivot）----
import matplotlib.pyplot as plt
from src.visualization.dashboard import plot_demand_overview

FIGURES_DIR = _root / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig = plot_demand_overview(demand_df, data_source=data_source,
                           save_path=FIGURES_DIR / "fig_quickstart_demand_overview.png")
plt.show()
print(f"图已保存: {FIGURES_DIR / 'fig_quickstart_demand_overview.png'}")

wide_df = pivot_demand_wide(demand_df)    # 长表 -> 宽表（index=date，columns=序列名）
wide_df.describe().round(2)               # 宽表视角的统计摘要

## 如何启用 M5 真实数据（三选一）

1. **手动放置（推荐）**：Kaggle 竞赛 `m5-forecasting-accuracy` 下载
   `sales_train_evaluation.csv`，放到 `data/raw/m5/` 下 → 把 Cell 3 的
   `USE_M5` 改为 `True` 重跑（daily 口径，自动截取前 365 天）。
2. **Kaggle API 自动下载**：设置 `KAGGLE_USERNAME` / `KAGGLE_KEY` 环境变量，
   框架 `load_m5()` 会自动下载并缓存到 `~/.deepbullwhip/data/m5/`。
3. **验证数据就位**：`pytest tests/test_forecaster.py -v` 里 M5 用例不再 skip 即成功。

> 注意：直接调框架 API 时必须显式传 `data_root="data/raw"`，否则 pip 安装下
> 默认路径指向 site-packages 内部必然 FileNotFoundError（对齐清单 §1.3）；
> 本项目的 `load_m5_demand` 已内置处理，无需关心。

## 小结与下一步
- 环境自检全 PASS = 环境就绪（Python 3.10/3.11 + deepbullwhip 0.4.1 + 依赖齐全）。
- 数据侧：`USE_M5=False` 时用合成 AR(1)（seed=42 可复现）开发不被阻塞；
  M5 CSV 就位后一键切换真实口径。
- 交接物：`ReplayDemandGenerator`（整表聚合 + 逐节点两种构造）、
  长表→宽表工具 `pivot_demand_wide` 均已验证。

**下一步** → `02_forecast_baseline.ipynb`：基线预测对比 + 自定义
WeightedMAForecaster + 误差指标落盘。